* Regression predicts a number (a price). Classification (Day 9) predicts a category (survived or not).
* Linear regression is the model you built on Day 6: ŷ = w·x + b. With several features it is ŷ = x @ w + b. 
* LinearRegression().fit(X, y) finds the best w and b for you. It solves for them directly, as np.polyfit does, instead of walking downhill step by step.
* The error of one prediction is e = y − ŷ. Four numbers summarise all the errors:

* Reading R²: 1 means perfect and 0 means no better than the mean. A negative R² means worse than the mean.
* Baseline for regression: DummyRegressor(strategy="mean") always predicts the training mean. Any real model must beat its MAE and RMSE.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

In [ ]:
# 1. look at the label
print(df.shape)
print(df["median_house_value"].describe().round(1).to_dict())
print((df["median_house_value"] >= 500001).sum())
print(round(df[["median_income", "median_house_value"]].corr().iloc[0, 1], 4))

(20640, 10)
{'count': 20640.0, 'mean': 206855.8, 'std': 115395.6, 'min': 14999.0, '25%': 119600.0, '50%': 179700.0, '75%': 264725.0, 'max': 500001.0}
965
0.6881


In [4]:
# 2. features, label, split
X = df[["median_income"]]
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(16512, 1) (4128, 1)


In [ ]:
# 3. a small helper to print the four metrics
def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:10s} MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}")

In [8]:
# 4. baseline
base = DummyRegressor(strategy="mean")
base.fit(X_train, y_train)
report("Baseline:", y_test, base.predict(X_test))

Baseline:  MAE=90,607  RMSE=114,486  R2=-0.0002


In [9]:
# 5. linear regression
model = LinearRegression()
model.fit(X_train, y_train)
print(round(model.coef_[0], 2), round(model.intercept_, 2))
pred = model.predict(X_test)
report("Linear:", y_test, pred)

41933.85 44459.73
Linear:    MAE=62,991  RMSE=84,209  R2=0.4589


In [11]:
# 6. predict two new districts (median income 3.0 and 5.0, in units of $10,000)
print(model.predict(pd.DataFrame({"median_income": [3.0, 5.0]}).round(0)))

[170261.27735052 254128.97613814]


In [13]:
# 7. compute the same metrics by hand
e = y_test - pred
print(round(np.mean(np.abs(e)), 1),
      round(np.mean(e ** 2), 1),
      round(1 - np.sum(e ** 2) / np.sum((y_test - y_test.mean()) ** 2), 4))

62990.9 7091157771.8 0.4589
